Check how the autoencoder outputs look across runs and seeds.  
The goal is to find high impact stable components.  Generally, another simple way is to simply take the minimum loss components from your elbow.  

But still, this notebook will show the components.  

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.mnist import SimpleMNIST
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
# from pt_to_api.benchmark.thresholding import threshold_assuming_noise_at_0_with_only_one_side_active
from pt_to_api.benchmark import thresholding as TR
import gc
import pickle
from pt_to_api.utils import otsu_threshold
from sklearn.preprocessing import normalize
from collections import defaultdict
import math
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import json
from dataclasses import dataclass
from pt_to_api import benchmark as B
from sklearn.mixture import GaussianMixture

In [ ]:
MODEL_PATH = Path("../../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../../pt-to-api/data/first-input-tens.pt")

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
DRIVE_PATH = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist")
MAIN_OUT_DIR = (DRIVE_PATH / "collect-patches" / "data")

layer_name = "layers.2"
channel = 0

In [ ]:
SingleRun = B.SingleRun
Autoencoder = B.Autoencoder
RunId = B.RunId
CompId = B.CompId
IdAndComp = B.IdAndComp
C2R = B.C2R



def _mse(x, codes, comps):
    diff = x - (codes@comps)
    return (diff**2).mean()


def get_comp_scores(X, codes, components):
    main_mse = _mse(X, codes, components)
    scores = []
    for i in range(len(components)):
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(X, codes, comps)
        comp_score = new_mse - main_mse
        scores.append(comp_score)
    return scores, main_mse


def get_comp_scores_when_active(X, codes, components, ):
    scores, active_ratios = [], []
    for i in range(len(components)):
        try:
            idxs = get_indices_where_comp_is_active(i, codes, components)
        except TR.ThresholdFailureException as ex:
            # now what?
            # we dont have indices for ith component
            # we would like to assign a score
            # in this case, we assign 0
            # we cant find the comp only so its fine
            print(f"WARN: could not find threshold for comp={i} total_components={len(components)}")
            print(f"\tcause: {ex}")
            print(f"\tsign result: {ex.sign_result}")
            scores.append(0)
            active_ratios.append(0)
            continue

        _X, _codes = X[idxs], codes[idxs]

        main_mse = _mse(_X, _codes, components)
        comps = components.copy()
        comps[i].fill(0)
        new_mse = _mse(_X, _codes, comps)
        comp_score = (new_mse - main_mse) / new_mse

        scores.append(comp_score)
        active_ratios.append(len(idxs) / codes.shape[0])
    return scores, active_ratios


def find_threshold_using_gmm(values, gap_ratio_to_add=0):
    # assume values are always positive
    if np.any(values < 0):
        raise Exception("values for finding threshold should always be positive")
    values = np.abs(np.array(values))
    gmm = GaussianMixture(n_components=2, random_state=0)
    gmm.fit(values.reshape(-1, 1))
    
    garbage_idx = np.argmin(gmm.means_)  # garbage is near 0, so smallest mean
    real_idx = 1 - garbage_idx
    
    midpoint = (gmm.means_[garbage_idx] + gmm.means_[real_idx]) / 2
    gap = gmm.means_[real_idx] - gmm.means_[garbage_idx]
    return (midpoint + gap_ratio_to_add * gap).item()

def get_indices_where_comp_is_active(comp_idx, codes, components):
    # it might be useful to actually persist these calculated thresholds
    comp = components[comp_idx]
    data = codes[:, comp_idx]
    thresh_state = TR.threshold_assuming_noise_at_0_with_only_one_side_active(data)
    thresh = None
    match thresh_state:
        case TR.Ambiguous() | TR.MixedSign():
            raise TR.ThresholdFailureException(f"failure in finding threshold comp_idx={comp_idx}", thresh_state)
        case TR.NonNegative(threshold = t): 
            thresh = t
        case TR.NonPositive(threshold = t):
            thresh = t
    if thresh is None:
        raise Exception("thresh cant be none, the match statement is not exhaustive")

    indices = np.argwhere(np.abs(data) >= np.abs(thresh))
    return indices

def get_weights_and_patches(layer_data_dir):
    weight = torch.load(layer_data_dir / "weight.pt", weights_only=False)
    patches = torch.load(layer_data_dir / "samples.pt", weights_only=False)
    return weight, patches

def get_loaded_normaliser(file_path):
    normaliser = B.NormaliseStdScaler()
    state = load_normaliser_state(file_path)
    normaliser.global_std_ = state["global_std_"]
    return normaliser

def load_normaliser_state(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

def load_samples_used_for_training(layer_data_dir):
    weight, patches = get_weights_and_patches(layer_data_dir)
    scaler = get_loaded_normaliser(layer_data_dir / "normaliser.json")
    pw = weight * patches
    scaled_pw = scaler.transform(pw)
    return scaled_pw, scaler


def load_c2r(runs_dir: Path) -> C2R:
    c2r = {}
    for p in runs_dir.rglob("*.pt"):
        seed = int(p.stem.split("_")[1])
        n_components = int(p.parent.stem)
        c2r[RunId(n_components, seed)] = torch.load(p, weights_only=False)
    return c2r

def get_serialised_components_and_their_lookup_table(c2r: C2R) -> tuple[list[np.ndarray], dict[int, CompId]]:
    idx_by_comp_id = {}
    all_comps = []
    for run_id, run in c2r.items():
        for comp_idx, comp in enumerate(run.components):
            idx = len(all_comps)
            all_comps.append(comp)
            idx_by_comp_id[idx] = CompId(run_id, comp_idx)
    return all_comps, idx_by_comp_id

def get_distance_matrix(all_comps: list[np.ndarray]) -> np.ndarray:
    X_reduced = normalize(all_comps, "l2")
    abs_sim = np.abs(cosine_similarity(X_reduced, X_reduced))  # (n_samples, n_samples)
    distance_matrix = 1 - abs_sim  # values in [0, 1]
    return distance_matrix


def get_comp2score(c2r: C2R, scaled_pw: np.ndarray) -> dict[CompId, float]:
    comp2score_when_active = {}
    for run_id, run in c2r.items():
        scores, active_ratios = get_comp_scores_when_active(scaled_pw, run.codes, run.components)
        scores = np.array(scores)
        for j in range(len(scores)):
            comp2score_when_active[CompId(run_id, j)] = (scores[j], active_ratios[j])
    return comp2score_when_active


def get_label_by_comps(
    cluster_labels: list[int], all_comps: list[np.ndarray], idx_by_comp_id: dict[int, CompId]
) -> dict[str, list[IdAndComp]]:
    l2comps = defaultdict(list)
    for i, label in enumerate(cluster_labels):
        comp_id = idx_by_comp_id[i]
        l2comps[label].append(IdAndComp(comp_id, all_comps[i]))
    return l2comps


def get_label_by_highest_scored_comp(
    label_by_comps, comp2score
):
    lable2comp = {}
    for label, id_and_comp_list in label_by_comps.items():
        scores = [
            comp2score[comp_id]
            for comp_id, _ in id_and_comp_list
        ]
        max_score_idx = np.argmax(scores)
        max_score = scores[max_score_idx]
        lable2comp[label] = (id_and_comp_list[max_score_idx], max_score)
    return label2comp

def show_label_by_comps(label_by_comps, comp2score, comp_shape):
    for label, id_and_comp_list in label_by_comps.items():
        # print("hahahaha", id_and_comp_list)
        comps, titles, scores = [], [], []
        for id_and_comp in id_and_comp_list:
            comp_id, comp = id_and_comp.comp_id, id_and_comp.comp
            comps.append(comp.reshape(comp_shape))
            score, active_ratio = comp2score[comp_id]
            titles.append(f"{comp_id.run_id.n_components} ({score:.3f}/{active_ratio:.3f})")
            scores.append(score)
        p50_score = np.median(scores)
        p75_score = np.percentile(scores, 75)
        cols = min(len(comps), 10)
        rows = math.ceil(len(comps) / cols)
        S(comps, (20,rows*3), cols, suptitle=f"{label} p50: {p50_score:.4f} p75: {p75_score:.4f}", ax_titles=titles)
        plt.show()


def plot_losses(c2r: C2R) -> None:
    comps_list = sorted(set([c.n_components for c in c2r]))
    losses = []
    for n in comps_list:
        min_loss_of_comp = np.min([c2r[RunId(n, seed)].loss for seed in [0,1,2]])
        losses.append(min_loss_of_comp)
    plt.plot(comps_list, losses)
    plt.show()

# Check comp-wise

In [ ]:
layer_name, channel = "layers.2", 0

layer_data_dir = MAIN_OUT_DIR / layer_name / str(channel)
runs_dir =  layer_data_dir / "runs"
SHAPE = (8,9)

c2r = load_c2r(runs_dir)

In [ ]:
from collections import defaultdict

n_components_list = sorted(set([c.n_components for c in c2r]))
n2seeds = defaultdict(list)
for c in c2r:
    n2seeds[c.n_components].append(c.seed)

n_components_list, n2seeds

In [ ]:
# 27 is awful
scaled_samples, scaler = load_samples_used_for_training(layer_data_dir)
comp2score = get_comp2score(c2r, scaled_samples)

### Visualise

We dont do > 15 components, they have many dead atoms. The loss curve is also quite bad.  
This component has quite unstable losses compared to K12.  

n_components = 14 looks fine.  

It is mainly a symptom of them being dead atoms.  
I might need better handling for checking dead atoms. The code currently is slightly hand wavy. But its okay.   

In [ ]:
plot_losses(c2r)

In [ ]:
n_components_list = [n for n in n_components_list if n < 15]

In [ ]:
for n in n_components_list:
    print("#############", n)
    for seed in n2seeds[n]:
        run = c2r[RunId(n, seed)]
        cols = min(len(run.components), 6)
        row_sz = math.ceil(len(run.components) / cols)*3
        ax_titles = []
        for i in range(len(run.components)):
            score, active_ratio = comp2score[CompId(RunId(n, seed), i)]  
            ax_titles.append(f"{score:.3f}/{active_ratio:.3f}")
        S(
            [c.reshape(SHAPE) for c in run.components],
            (20, row_sz),
            cols,
            ax_titles=ax_titles,
            suptitle=f"{seed}: {run.loss}",
        )
        plt.show()

# Cluster the components

If you cluster components across all `n_components`, it can get confusing quite fast.  
It is better to find the range which is useful for analysis. And only cluster there.  


So we do a range. And its important to also remove the dead atoms. For now, we use score == 0, although this is brittle.  
But then, it also can make sense, a dead atom has zero score :)   


In [ ]:
filtered_c2r = {}
allowed_n_comps = [10, 11, 12, 13, 14]
for run_id, run in c2r.items():
    if run_id.n_components not in allowed_n_comps:
        continue
    filtered_c2r[run_id] = run
    

    ######### we dont remove dead atoms for now, they go to -1 anyways
    # for i, comp in enumerate(run.components):
    #     comp_id = CompId(run_id, i)
    #     score, _ = comp2score[comp_id]
    #     if score == 0:
    #         # dead atom
    #         continue

In [ ]:
all_comps, idx_by_comp_id = get_serialised_components_and_their_lookup_table(filtered_c2r)

In [ ]:
distance_matrix = get_distance_matrix(all_comps)
hdbscan = HDBSCAN(copy=True, min_cluster_size=5, metric="precomputed")
hdbscan.fit(distance_matrix)

In [ ]:
scaled_samples, scaler = load_samples_used_for_training(layer_data_dir)

In [ ]:
label_by_comps = get_label_by_comps(hdbscan.labels_, all_comps, idx_by_comp_id)
comp2score = get_comp2score(filtered_c2r, scaled_samples)

In [ ]:
label_by_comps

In [ ]:
show_label_by_comps(label_by_comps, comp2score, (8,9))

We are in the territory of having a lot of components which are hard to look at manually. and verify what works :).   
Well kodewa kodewa this sucks.  
It might be best to train the model with comps initialised from each cluster and see which ones remain alive.  
Easy, peasy.  

Some components are not stable, and they are not high contributing.  
Lets see if we can think of a simple metric.


- First: if the `score` itself, which is now quite representative (the amount of MSE difference when the component was active) is a good indicator.  
  - We need to find the ones which score very low.  
  - Note that score finding itself is slightly mathy (it has a formula). I dont know if I can naively divide something from it.  
  - Looking at the seeds, it is clear that training did not go very well for some of them. We NEED fast seed evaluation now. I can run mass seed evals, and only use the ones which are near the minima, whatever near means.  
  - How do we do that though? I might use something like a GMM again for this (GMM thresholding FTW, the num components can be 1 or 2 now though, will need to check).  
    - For now, manual, but I'll need to test approaches.
    - The first thing is automating seed runs though, its too expensive to do multiple seeds right now.  
- Anyways, this is for reducing the components, making each run more stable. We still have the problem of what to do right now.  
  - components can have overlaps. We want to find the components which "remain" in the model after training. others would drift.  
  - now which conmponent to pick from the cluster. 
    - highest score?
    - highest similarity? -> this makes most sense
    - cluster centroid? -> might not be the best bet, im not sure.  
  - similarity seems to be what im doing. Maybe that is good enough.  

In [ ]:
def get_medoids(distance_matrix, labels):
    medoids = {}
    for label in set(labels):
        if label == -1:
            continue
        mask = np.where(labels == label)[0]
        sub_matrix = distance_matrix[np.ix_(mask, mask)]
        medoid_local_idx = sub_matrix.sum(axis=1).argmin()
        medoids[label] = mask[medoid_local_idx]
    return medoids  # original indices

In [ ]:
medoids = get_medoids(distance_matrix, hdbscan.labels_)

In [ ]:
# l2comp = {label: all_comps[comp_idx] for label, comp_idx in medoids.items()}

In [ ]:
comps = []
titles = []
for label, comp_idx in medoids.items():
    comp = all_comps[comp_idx]
    comps.append(comp.reshape(SHAPE))
    comp_id = idx_by_comp_id[comp_idx]
    score, active_ratio = comp2score[comp_id]
    titles.append(f"L({label}) ({score:.3f}/{active_ratio:.3f})")

In [ ]:
# checked, these are perfect.
S(comps, (20, 9), 6, ax_titles=titles)
plt.show()

In [ ]:
scaled_samples.shape, np.array(comps).shape

In [ ]:
torch.mps.empty_cache()
gc.collect()

In [ ]:
# now we train a model with these medoids as initialsations for the decoder.  
# everything else goes normally
# we specifically init the encoder with sigma_enc separately.
# this requires some work which ive done before, but oh well.
from pt_to_api.benchmark import train_x as TX
from pt_to_api import benchmark as B

comp_arr = np.array([c.reshape(-1) for c in comps]).copy()
model = TX.Autoencoder(scaled_samples.shape[1], comp_arr.shape[0])
model.decoder.weight.data = torch.tensor(comp_arr.T)

run = TX.train(
    scaled_samples,
    comp_arr.shape[0],
    1e-2,
    device="mps",
    epochs=1000,
    baseline_epochs=600,
    batch_size=64,
    use_ln_term=False,
    recon_err_schedule=TX.CosineAnnealReconError(1000),
    initialised_model=model,
    init_strategy=B.OnlyInitEncoderStrategy(),
)

In [ ]:
torch.save(run, "./run-random-weights-algo.pt")
torch.save(comp_arr, "./original-clustered-comps.pt")

In [ ]:
# if we look at the point of these operations, the main point below is to remove L(7) and L(8) competetion
# and realising the L(6) is shit
# Why is L(6) bad? it already does not explain much, so descent would replace it with something better by decomposing
# Why do L(7) and L(8) change? Cuz they are overlapping (technically, they are just the same things)

# currently, Im running the model again to do this. but is that wise?

# i keep coming back to the fact that i can simply select the min loss thing at some components. check out many seeds which are close at loss
# and keep the components which are stable. and train a new model with those initialised.
# my current problem is me going through clustering multiple n-comps, which might not be the best solution
# i guess we back to simply using good seeds
# now the problem right now is i need more seeds, im not getting stable losses.  
# so the next step does seem like ill need to work on JAX

In [ ]:
# original, show again
S(comps, (20, 9), 6, ax_titles=titles)
plt.show()

In [ ]:
# the main problem is splitting. how do we encourage no splitting?
# i do see what all is going away, but the decomposition is quite annoying.
# there should be no decomposition, only going to 0 or not
# what if we train only the encoder? 
# the rationale being that the components which are bigger forms of some other component would have overlaps.
# and as such, wont be used?
S([c.reshape(SHAPE) for c in run.components], (20, 9), 6, ax_titles=[])
plt.show()

In [ ]:
run.components.shape, np.array(comps).shape

In [ ]:
np.concat([run.components, np.array([c.reshape(-1) for c in comps])])

In [ ]:
distance_matrix = get_distance_matrix(all_comps)
hdbscan = HDBSCAN(copy=True, min_cluster_size=5, metric="precomputed")
hdbscan.fit(distance_matrix)